# Dialforge Final Acceptance

This is the **final synthetic production acceptance benchmark** for the hardened Dialforge caller. It tests the warmed path that matters during a real conversation, not just cold model startup.

It measures **Qwen 3 1.7B / 4B / 8B**, production **faster-whisper small.en int8**, prewarmed **Chatterbox Nano**, chunk-safe reasoning guards, action-integrity behavior, STT accuracy, TTS real-time factor, GPU pressure, and perceived **time to first playable spoken response**.

The headline score uses **Qwen 3 4B**, Dialforge's recommended 8 GB-class production tier. A `PHENOMENAL` verdict is only awarded if the measured score actually earns it.

### Run it
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Click **Runtime → Run all**.
3. Leave the tab open until the final report appears.

The notebook pins itself to one immutable Git commit, so the bootstrap and benchmark runner cannot drift during the run. No SIP credentials or real phone calls are used.

In [ ]:
# ONE-CLICK DIALFORGE FINAL ACCEPTANCE
import base64, json, os, pathlib, subprocess, sys, time, urllib.request
from IPython.display import HTML, FileLink, display

OWNER = 'SumamaAhmed69'
REPO = 'Axemetric-Caller-Beta-Runtime'
BOOTSTRAP_PATH = 'benchmarks/dialforge_final_colab_bootstrap.py'
BOOTSTRAP = pathlib.Path('/content/dialforge_final_colab_bootstrap.py')
REPORT = pathlib.Path('/content/dialforge-final-acceptance/dialforge-final-acceptance.html')
JSON_REPORT = pathlib.Path('/content/dialforge-final-acceptance/dialforge-final-acceptance.json')
ZIP_REPORT = pathlib.Path('/content/dialforge-final-acceptance-results.zip')

def api_json(url):
    req = urllib.request.Request(url, headers={'User-Agent':'Dialforge-Final-Colab','Accept':'application/vnd.github+json'})
    with urllib.request.urlopen(req, timeout=60) as response:
        return json.loads(response.read().decode('utf-8'))

print('Resolving one immutable Dialforge source revision...')
source_sha = api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/main?x={time.time_ns()}')['sha']
print('Pinned source:', source_sha)

last_error = None
for attempt in range(1, 6):
    try:
        item = api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/contents/{BOOTSTRAP_PATH}?ref={source_sha}&x={time.time_ns()}')
        payload = base64.b64decode(item['content'])
        text = payload.decode('utf-8')
        compile(text, str(BOOTSTRAP), 'exec')
        BOOTSTRAP.write_bytes(payload)
        print(f'Bootstrap verified: {len(payload)} bytes')
        break
    except Exception as exc:
        last_error = exc
        print(f'Bootstrap download attempt {attempt}/5 failed: {exc}')
        time.sleep(2 * attempt)
else:
    raise RuntimeError(f'Could not download final Dialforge bootstrap: {last_error}')

env = os.environ.copy()
env['DIALFORGE_SOURCE_SHA'] = source_sha
print('\nStarting Dialforge Final Acceptance. This can take a while on the first clean Colab VM...')
result = subprocess.run([sys.executable, str(BOOTSTRAP)], env=env)
if result.returncode != 0:
    raise RuntimeError('Final acceptance stopped. The exact failing component is printed above.')
if not REPORT.exists() or not JSON_REPORT.exists() or not ZIP_REPORT.exists():
    raise RuntimeError('Final acceptance completed without all report files.')

print('\n=== DIALFORGE FINAL ACCEPTANCE COMPLETE ===')
display(HTML(REPORT.read_text(encoding='utf-8')))
print('\nDownload/share these results:')
display(FileLink(str(ZIP_REPORT)))
display(FileLink(str(JSON_REPORT)))


## What the verdict means

`PHENOMENAL` is deliberately difficult to earn: the final score must be at least 92/100, all quality and action-integrity scenarios must pass, the warmed median first-playable-response time must be 3 seconds or less, and the guard suite must have no blocker.

This remains a **synthetic local-AI benchmark**. LiveKit, SIP carrier and PSTN network latency are not included. After this passes, one real Windows SIP call is still the final telephony acceptance test.